In [ ]:
# I have trained the LSTM model on the same model which we later train the attention model on ,
# so that i can get a vague sense of superiority of the attention model over this model.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [51]:
with open('/home/kshu/soc26/SOC-26-token_to_translations-/Week4/input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
# Hyperparameters
batch_size = 321
block_size = 8       
embed_dim = 64
hidden_dim = 256
num_layers = 2
dropout = 0.1
learning_rate = 3e-4
max_iters = 50001
eval_interval = 500
eval_iters = 100
device = 'cuda' if torch.cuda.is_available() else 'cpu'
 
torch.manual_seed(1337)

In [53]:
vocab = sorted(list(set(text)))
vocab_size = len(vocab)
stoi = {ch: i for i, ch in enumerate(vocab)}
itos = {i: ch for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])


# make train and test split


text = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(text))
train_data = text[:n]
val_data = text[n:]

In [54]:
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y



In [55]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            x, y = get_batch(split)
            _, loss = model(x, y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

In [56]:
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Linear(hidden_dim, vocab_size)
 
    def forward(self, idx, targets=None):
        x = self.embedding(idx)              # (B, T, embed_dim)
        out, _ = self.lstm(x)                # (B, T, hidden_dim)
        logits = self.head(out)              # (B, T, vocab_size)
 
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss
 
    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]      
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]           
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        self.train()
        return idx

In [57]:
model = LSTMLanguageModel(vocab_size, embed_dim, hidden_dim, num_layers, dropout).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [62]:
learning_rate=3e-5

In [63]:
# train 

 
for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
 
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 1.7516, val loss 1.8651
step 500: train loss 1.7440, val loss 1.8741
step 1000: train loss 1.7318, val loss 1.8734
step 1500: train loss 1.7450, val loss 1.8570
step 2000: train loss 1.7270, val loss 1.8440
step 2500: train loss 1.7252, val loss 1.8812
step 3000: train loss 1.7267, val loss 1.8479
step 3500: train loss 1.7118, val loss 1.8448
step 4000: train loss 1.7156, val loss 1.8547
step 4500: train loss 1.7090, val loss 1.8457
step 4999: train loss 1.7040, val loss 1.8460


In [59]:
# generate 
context = torch.zeros((1, 1), dtype=torch.long, device=device) 
generated = model.generate(context, max_new_tokens=500)[0].tolist()
print(decode(generated))



COMINILA:
Ocown,
The lay be madiest bube to take Ond my dagatanss:
Whith foul he hadt?
Feetlessaness.

HENRET:
Dod, when must of the of it heart my follling egrient:
In contlatiHer drove to and Wall.

DUKE Wabous less die; let hus qouch by stand aissell, yet love.
I camopetelives
Morthy worly thake on in on her evicks to them srive and him he poor of he reshepk of thrupt for treary tome to tey.

MUEET:
Nay Pring my of.

HENRY ESCALUS:
Your adsal the Earth, hoin cour ay and your tome from care,

